# YouTube Relaxing Music Video Generator
Generate 10-minute relaxing videos with background footage and real audio

In [ ]:
# Install dependencies
!pip install requests python-dotenv -q

In [ ]:
import os, requests, subprocess, random
from pathlib import Path

OUTPUT = Path('/content/output')
OUTPUT.mkdir(parents=True, exist_ok=True)
AUDIO = OUTPUT / 'ambient.mp3'
VIDEO = OUTPUT / 'relaxing_video.mp4'

In [ ]:
# Generate ambient audio (nature sounds)
def generate_audio():
    print('Generating ambient audio...')
    cmd = [
        'ffmpeg', '-y',
        '-f', 'lavfi', '-i', 'anoise=amount=0.05:color=brown:duration=600',
        '-c:a', 'libmp3lame', '-b:a', '128k',
        str(AUDIO)
    ]
    subprocess.run(cmd, capture_output=True)
    return AUDIO.exists()

In [ ]:
# Download stock footage from Pexels
def download_stock(query='nature relaxation', count=3):
    api_key = os.environ.get('PEXELS_API_KEY', '')
    if not api_key:
        print('No Pexels API key - will use solid color background')
        return []
    
    print(f'Downloading stock footage: {query}')
    headers = {'Authorization': api_key}
    resp = requests.get(
        'https://api.pexels.com/v1/search',
        headers=headers,
        params={'query': query, 'per_page': count, 'orientation': 'landscape'}
    )
    
    videos = []
    if resp.status_code == 200:
        for v in resp.json().get('videos', [])[:count]:
            video_url = v['video_files'][0]['link']
            out = OUTPUT / f'stock_{len(videos)}.mp4'
            try:
                video_data = requests.get(video_url, timeout=60).content
                out.write_bytes(video_data)
                videos.append(str(out))
                print(f'  Downloaded: {v.get("id", "video")}')
            except Exception as e:
                print(f'  Error: {e}')
    return videos

In [ ]:
# Create video with audio
def create_video():
    print('\n=== Creating Relaxing Video with Audio ===')
    
    # Generate audio
    generate_audio()
    
    # Download stock footage
    stock = download_stock('peaceful nature', 3)
    
    if stock:
        print(f'\nUsing {len(stock)} stock videos')
        
        # Concatenate stock videos
        list_file = OUTPUT / 'list.txt'
        list_file.write_text('\n'.join(f"file '{s}'" for s in stock))
        
        cmd = [
            'ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', str(list_file),
            '-i', str(AUDIO),
            '-c:v', 'libx264', '-preset', 'medium', '-crf', '23',
            '-c:a', 'aac', '-b:a', '128k',
            '-pix_fmt', 'yuv420p', '-t', '600',
            str(VIDEO)
        ]
    else:
        print('\nNo stock footage - creating video with animated background')
        
        cmd = [
            'ffmpeg', '-y',
            '-f', 'lavfi', '-i',
            f'color=c=#0a1a2e:s=1920x1080:d=600:rate=30',
            '-vf',
            "drawtext=fontfile=/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf:text='Relaxing Music for Stress Relief':fontsize=48:fontcolor=white:x=(w-text_w)/2:y=h/3,"\
            "drawtext=fontfile=/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf:text='Peaceful Ambient Sounds':fontsize=28:fontcolor=#aaa:x=(w-text_w)/2:y=2h/3",
            '-i', str(AUDIO),
            '-c:v', 'libx264', '-preset', 'medium', '-crf', '25',
            '-c:a', 'aac', '-b:a', '128k',
            '-pix_fmt', 'yuv420p',
            str(VIDEO)
        ]
    
    print('Rendering video... (this may take a few minutes)')
    subprocess.run(cmd, capture_output=True)
    
    if VIDEO.exists() and VIDEO.stat().st_size > 100000:
        size_mb = VIDEO.stat().st_size / 1024 / 1024
        print(f'\nSUCCESS! Video: {size_mb:.2f} MB, 10 minutes, 1920x1080')
        return str(VIDEO)
    else:
        print('ERROR: Video generation failed')
        return None

In [ ]:
# Generate the video
video_path = create_video()
print(f'\nVideo saved to: {video_path}')

In [ ]:
# Download the video
from google.colab import files
files.download(video_path)